# Lab 5: Robo-advisers

The fee table, then the four-ETF split

A household has £10,000 to invest. A
[robo-adviser](../resources/glossary.qmd#robo-adviser) may make advice
affordable, but its portfolio still depends on estimated returns and
risks. Use the fee calculation to explain access, then test whether an
allocation chosen from earlier data performs well on later data.

Work on one four-ETF portfolio throughout: SPY, EFA, BND and TLT. SPY
holds US shares; EFA holds shares in developed markets outside the US
and Canada; BND holds a broad US bond portfolio; TLT holds long-dated US
Treasury bonds. An [ETF](../resources/glossary.qmd#etf) is an
exchange-traded fund. These US-listed funds are a teaching dataset,
rather than a recommendation for a UK household.

Tasks 1 and 2 are the main in-class work, taking about 10 and 25
minutes. Tasks 3 and 4 follow the lecture demonstrations: trace one
training/test step and one resampling step in class, then finish the
runs after class if needed. Reserve 10 minutes for the explanation in
Task 5. There is no Blackboard upload. The covariance and shrinkage
exercises at the end are optional practice.

FIN510 uses the course parquet in Colab. FIN306 can extract the same
four ETFs in the Financial Innovation Lab in semester 2 and rerun the
analysis on that extract.

<figure>
<a
href="https://colab.research.google.com/github/quinfer/fin510-colab-notebooks/blob/main/labs/lab05_robo.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open in Colab</figcaption>
</figure>

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import minimize

for _module_dir in ("scripts", "../scripts", "resources", "../resources"):
    _module_path = Path(_module_dir).resolve()
    if (_module_path / "bloomberg_loader.py").is_file():
        sys.path.insert(0, str(_module_path))
        break
try:
    from bloomberg_loader import load_bloomberg
except ImportError:
    def load_bloomberg(tickers=None):
        raise FileNotFoundError("Shared loader unavailable; use the Colab data fallback.")

## Task 1: The floor binds

Use the illustrative tariff from the lecture: traditional advice costs
1.5% of the account, subject to a minimum annual fee of £2,500. The robo
tariff is 0.25% with no minimum fee. These are teaching assumptions,
rather than current provider quotes.

For account value $A$,

$$\text{traditional fee}=\max(0.015A,2500), \qquad
\text{effective rate}=\frac{\text{annual fee}}{A}.$$

The [fee-floor threshold](../resources/glossary.qmd#fee-floor-threshold)
is the account value at which the percentage fee equals the minimum:

$$0.015A=2500\quad\Rightarrow\quad A=2500/0.015\approx £166{,}667.$$

This is the breakpoint of a piecewise-linear fee schedule. Below it the
minimum sets the fee; above it the percentage charge sets the fee. The
two advice services do not charge the same amount at this threshold.

Before running the cell, calculate both fees for £10,000. Which part of
the traditional rule applies?

In [ ]:
trad_rate = 0.015
trad_floor = 2500
robo_rate = 0.0025
table_sizes = np.array([10_000, 25_000, 50_000, 100_000, 250_000, 500_000, 1_000_000])
table_trad = np.maximum(table_sizes * trad_rate, trad_floor)
table_robo = table_sizes * robo_rate

print(f"{'Account':>12} {'Trad £':>10} {'Trad %':>8} {'Robo £':>10} {'Robo %':>8}")
for size, trad, robo in zip(table_sizes, table_trad, table_robo):
    print(
        f"£{size:>11,} {trad:10,.0f} {trad/size*100:7.1f}% "
        f"{robo:10,.2f} {robo/size*100:7.2f}%"
    )
print(f"\nFloor binds below £{trad_floor / trad_rate:,.0f}")

In [ ]:
sizes = np.linspace(10_000, 1_000_000, 400)
trad_fee = np.maximum(sizes * trad_rate, trad_floor)
robo_fee = sizes * robo_rate
fig, axes = plt.subplots(1, 2, figsize=(10, 4.4), sharex=True)
x = sizes / 1000
c_trad, c_robo = "#4C78A8", "#E45756"
axes[0].plot(x, trad_fee, color=c_trad, lw=2, label="Traditional adviser")
axes[0].plot(x, robo_fee, color=c_robo, lw=2, label="Robo-adviser")
axes[0].scatter(table_sizes / 1000, table_trad, color=c_trad, s=28, zorder=3)
axes[0].scatter(table_sizes / 1000, table_robo, color=c_robo, s=28, zorder=3)
axes[0].set_ylabel("Annual fee (£)")
axes[0].yaxis.set_major_formatter(lambda v, _: f"{v:,.0f}")
axes[0].legend(frameon=False, fontsize=8)
axes[1].plot(x, trad_fee / sizes * 100, color=c_trad, lw=2, label="Traditional adviser")
axes[1].plot(x, robo_fee / sizes * 100, color=c_robo, lw=2, label="Robo-adviser")
axes[1].scatter(table_sizes / 1000, table_trad / table_sizes * 100, color=c_trad, s=28, zorder=3)
axes[1].scatter(table_sizes / 1000, table_robo / table_sizes * 100, color=c_robo, s=28, zorder=3)
axes[1].set_ylabel("Effective rate (%)")
axes[1].legend(frameon=False, fontsize=8)
for ax in axes:
    ax.set_xlabel("Account size (£000s)")
    ax.set_xlim(0, 1000)
fig.tight_layout()
plt.show()

**Your task:** explain the £10,000 row in two sentences. Show the
effective-rate calculation and explain why a minimum fee can prevent
access. Then use Reher and Sokolinski (2024) to distinguish a change in
participation from an estimated welfare gain. State the denominator for
each percentage.

## Task 2: Four ETFs, one split

Estimate the [max-Sharpe](../resources/glossary.qmd#sharpe-ratio)
allocation using returns dated before **2 July 2021**, then evaluate it
on returns from that date onwards. Compare it with
[1/N](../resources/glossary.qmd#one-over-n), which allocates 25% to each
fund, and a 60% SPY / 40% BND benchmark. The 40% cap applies to the
optimiser; the 60/40 benchmark deliberately uses a different rule.

The data are Bloomberg `PX_LAST` prices. Percentage changes measure
price returns, excluding cash distributions; these are not total
investment returns. All calculations use the same convention, an
illustrative 2% annual risk-free rate and no fees or trading costs.

The expression `returns @ weights` applies the same target weights each
day. It represents frictionless daily rebalancing to those targets,
rather than a buy-and-hold portfolio whose weights drift. In Task 2 the
targets are fixed after estimation; Task 3 updates them every 21 trading
days.

Before running the solver, predict which result would be convincing
evidence: a high fitted Sharpe, or a high Sharpe on the later sample?

In [ ]:
tickers = ["SPY", "EFA", "BND", "TLT"]
try:
    bbg = load_bloomberg(tickers=tickers)
except FileNotFoundError:
    bbg = pd.read_parquet(
        "https://raw.githubusercontent.com/quinfer/fin510-colab-notebooks/main/resources/bloomberg_database.parquet"
    )
    bbg = bbg[bbg["ticker"].isin(tickers)]

px = (
    bbg.assign(date=lambda d: pd.to_datetime(d["date"]))
    .pivot_table(index="date", columns="ticker", values="PX_LAST")
    .sort_index()
    [tickers]
    .dropna()
)
r = px.pct_change(fill_method=None).dropna()
split = pd.Timestamp("2021-07-02")
r_is = r.loc[r.index < split]
r_oos = r.loc[r.index >= split]
if r_is.empty or r_oos.empty:
    raise ValueError("The dataset must include returns before and after 2 July 2021.")
assert r_is.index.max() < r_oos.index.min()
print(f"Price sample: {px.index.min().date()} to {px.index.max().date()}")
print(f"Training ends: {r_is.index.max().date()} ({len(r_is)} returns)")
print(f"Testing starts: {r_oos.index.min().date()} ({len(r_oos)} returns)")

In [ ]:
rf = 0.02
n = len(tickers)

def rule_stats(w, rets):
    p = rets @ w
    ann = float(p.mean() * 252)
    vol = float(p.std() * np.sqrt(252))
    sh = (ann - rf) / vol if vol > 0 else np.nan
    return ann, vol, sh

def max_sharpe_weights(rets):
    n_assets = rets.shape[1]
    mu = rets.mean() * 252
    cov = rets.cov() * 252
    def neg_sh(w):
        vol = np.sqrt(w @ cov @ w)
        if vol == 0:
            return np.inf
        return -((w @ mu) - rf) / vol
    res = minimize(
        neg_sh, np.ones(n_assets) / n_assets, method="SLSQP",
        bounds=[(0, 0.4)] * n_assets,
        constraints={"type": "eq", "fun": lambda w: w.sum() - 1},
        options={"maxiter": 1000, "ftol": 1e-9},
    )
    feasible = (
        np.isclose(res.x.sum(), 1, atol=1e-7)
        and np.all(res.x >= -1e-7)
        and np.all(res.x <= 0.4 + 1e-7)
    )
    if not res.success or not feasible or not np.isfinite(res.fun):
        raise RuntimeError(f"Portfolio solve failed: {res.message}")
    return res.x

w_eq = np.ones(n) / n
w_6040 = np.array([0.6 if t == "SPY" else 0.4 if t == "BND" else 0.0 for t in tickers])
w_opt = max_sharpe_weights(r_is)
rules = [("1/N", w_eq), ("60/40", w_6040), ("Max Sharpe", w_opt)]

print("Weights (SPY, EFA, BND, TLT):")
for name, w in rules:
    print(f"  {name:11} " + " ".join(f"{t} {wi*100:5.1f}%" for t, wi in zip(tickers, w)))

print(f"\n{'Rule':<12} {'IS Sh':>7} {'Later Sh':>9}")
for name, w in rules:
    _, _, s0 = rule_stats(w, r_is)
    _, _, s1 = rule_stats(w, r_oos)
    print(f"{name:<12} {s0:7.2f} {s1:9.2f}")

**Your task:** connect the allocations to the subsequent results.

1.  Which allocation has the highest fitted Sharpe? Does it also have
    the highest later-sample Sharpe?
2.  Compare the TLT weights. TLT’s long-dated bonds are sensitive to
    interest-rate changes, an exposure called duration. How does that
    help explain the later comparison? BND also has interest-rate risk,
    so holding no TLT does not remove all duration exposure.
3.  Give one reason this comparison cannot establish that one allocation
    rule will always win.

Use the results your cells print. If a later data extract changes the
figures, report its dates and explain the result rather than copying the
lecture values.

## Task 3: Walk-forward, not a shuffle

A single split tests one set of estimated targets. [Walk-forward
validation](../resources/glossary.qmd#walk-forward-validation) repeats
the estimation as time moves forward. Use 252 trading days to estimate
the allocation, apply it to the following 21 days, then move both
windows forward by 21 days.

Trace the first iteration with your neighbour before running the full
loop. `train` ends immediately before `test` begins. A random shuffle
would let future observations enter an earlier training set.

The targets remain fixed within each 21-day test block. As in Task 2,
daily returns use frictionless rebalancing to those targets. This run
charges no costs. Compare both rules on precisely the same out-of-sample
dates.

In [ ]:
def rolling_backtest(returns, train_window=252, test_window=21):
    oos_opt, oos_eq, dates = [], [], []
    n_assets = returns.shape[1]
    for t in range(train_window, len(returns), test_window):
        train = returns.iloc[t - train_window:t]
        test = returns.iloc[t:t + test_window]
        assert train.index.max() < test.index.min()
        w = max_sharpe_weights(train)
        oos_opt.extend((test @ w).values)
        oos_eq.extend((test @ (np.ones(n_assets) / n_assets)).values)
        dates.extend(test.index)
    if not dates:
        raise ValueError("Not enough observations for a training and test window.")
    opt = pd.Series(oos_opt, index=dates)
    eq = pd.Series(oos_eq, index=dates)
    def sharpe(s):
        vol = s.std() * np.sqrt(252)
        return (s.mean() * 252 - rf) / vol if vol > 0 else np.nan
    print(f"Out-of-sample: {opt.index.min().date()} to {opt.index.max().date()}")
    print(f"Out-of-sample days: {len(opt)}")
    print(f"Capped max-Sharpe Sharpe: {sharpe(opt):.3f}")
    print(f"1/N Sharpe:               {sharpe(eq):.3f}")
    fig, ax = plt.subplots(figsize=(10, 4.2))
    ax.plot((1 + opt).cumprod(), label="Capped max-Sharpe", color="#4C78A8", lw=2)
    ax.plot((1 + eq).cumprod(), label="Equal-weight", color="0.5", lw=2, ls="--")
    ax.set_ylabel("Growth of 1 unit")
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()
    return pd.DataFrame({"Capped max-Sharpe": opt, "1/N": eq})

walk_forward_returns = rolling_backtest(r)

**Your task:** report the two Sharpes and write a conclusion qualified
by the sample, price-return convention and omission of costs. Explain
why a win on this path can coexist with DeMiguel, Garlappi, and Uppal
(2009)’s finding that optimised rules do not consistently outperform 1/N
across their comparisons. What information would you need to charge
trading costs?

## Task 4: Bootstrap the weights

Walk-forward asks how a rule performs on subsequent dates. The
[bootstrap](../resources/glossary.qmd#bootstrap) asks how sensitive the
estimated allocation is to the observed sample. Here, resample days from
the training period with replacement, re-estimate means and covariances,
and solve again. The four ETFs and 40% cap remain unchanged.

Trace one resample in class. A day may appear more than once or be
omitted. Sampling individual days breaks time dependence, including
volatility clustering, so interpret the resulting range as an
instability diagnostic rather than a calibrated confidence interval. The
plot shows the allocation, rather than another performance backtest.

In [ ]:
def bootstrap_weights(returns, n_boot=200, seed=42):
    rng = np.random.default_rng(seed)
    n_obs, n_assets = returns.shape
    W = []
    for _ in range(n_boot):
        idx = rng.integers(0, n_obs, size=n_obs)
        W.append(max_sharpe_weights(returns.iloc[idx]))
    W = np.array(W)
    print(f"{'Asset':<6} {'Mean':>8} {'2.5%':>8} {'97.5%':>8}")
    for i, ticker in enumerate(returns.columns):
        lo, hi = np.percentile(W[:, i], [2.5, 97.5])
        print(f"{ticker:<6} {W[:, i].mean()*100:7.1f}% {lo*100:7.1f}% {hi*100:7.1f}%")
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.boxplot(W * 100, whis=(2.5, 97.5))
    ax.set_xticks(range(1, n_assets + 1), returns.columns)
    ax.axhline(40, color="0.4", ls="--", label="40% cap")
    ax.set_ylabel("Estimated target weight (%)")
    ax.set_ylim(-2, 43)
    ax.legend(frameon=False)
    fig.tight_layout()
    plt.show()
    return W

resampled_weights = bootstrap_weights(r_is)

**Your task:** report the TLT 2.5th and 97.5th percentiles printed by
this run. Explain whether a single estimated weight adequately describes
the allocation’s stability. If resamples reach 40%, what can you infer
about the constraint, and what remains unknown about the unconstrained
solution?

## Task 5: Door, not mix

Return to the household with £10,000. It needs affordable access and an
allocation that remains defensible when estimates change. Automatic
rebalancing may also reduce discretionary trading, but its benefit needs
evidence of its own.

**Deliverable:** write five sentences that answer the following
questions. There is no Blackboard upload.

1.  What does the fee calculation tell the household about access?
2.  What changed in Reher and Sokolinski’s study, and how do their
    participation and welfare measures differ?
3.  Which allocation performed best on the later sample, and what
    exposure helps explain the result?
4.  What can your walk-forward and bootstrap results establish about
    performance and instability?
5.  Why can Barber and Odean’s heavy-trader comparison not be treated as
    the return a robo will add?

If you finish the full runs after class, update sentences 3 and 4 with
your own output.

## Optional practice: covariance

Use hypothetical stocks with 20% annual volatility and bonds with 10%
annual volatility. Allocate 50% to each. Keep their individual
volatilities fixed and change only their correlation $\rho$:

$$\sigma_p=\sqrt{0.5^2(0.20)^2+0.5^2(0.10)^2+2(0.5)(0.5)\rho(0.20)(0.10)}.$$

This two-asset illustration isolates covariance; its 50% weights do not
use the four-ETF optimiser’s 40% cap.

In [ ]:
for rho in (1.0, 0.5, 0.0):
    variance = 0.5**2 * 0.20**2 + 0.5**2 * 0.10**2 + 2 * 0.5 * 0.5 * rho * 0.20 * 0.10
    print(f"Correlation {rho:.1f}: portfolio volatility {np.sqrt(variance):.2%}")

**Your task:** explain why the portfolio volatility changes although
neither asset becomes less volatile.

## Optional practice: shrinkage

A fixed [shrinkage](../resources/glossary.qmd#shrinkage) rule pulls each
estimated return halfway towards their grand mean. These are
illustrative annual return estimates, rather than new ETF observations.

In [ ]:
sample_means = np.array([0.08, 0.04, 0.06])
prior_mean = sample_means.mean()
shrunk_means = 0.5 * sample_means + 0.5 * prior_mean
print("Grand mean:", f"{prior_mean:.1%}")
print("Shrunk estimates:", [f"{x:.1%}" for x in shrunk_means])

**Your task:** calculate the changes before running the cell. Explain
what the rule changes and what it leaves untouched. Why might a less
extreme estimate help, without guaranteeing a better realised portfolio
return?

FIN306 only: extract SPY, EFA, BND and TLT in the Financial Innovation
Lab in semester 2, then rerun Tasks 2–4 on that extract.

[Week 5 practice items](../practice/week05_robo.qmd): two written items
and twelve application questions, including the fee figure and the
four-ETF split. Not an Exercise 2 paper.

DeMiguel, Victor, Lorenzo Garlappi, and Raman Uppal. 2009. “Optimal
Versus Naive Diversification: How Inefficient Is the 1/n Portfolio
Strategy?” *The Review of Financial Studies* 22 (5): 1915–53.
<https://doi.org/10.1093/rfs/hhm075>.

Reher, Michael, and Stanislav Sokolinski. 2024. “Robo-Advisors and
Access to Wealth Management.” *Journal of Financial Economics* 155:
103829. <https://doi.org/10.1016/j.jfineco.2024.103829>.